# V20 – Datenbanken Teil 2: Theorie

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- die Grundidee der **Normalisierung** (1NF, 2NF, 3NF) erklären und Redundanzen erkennen,
- **Fremdschlüssel** sinnvoll einsetzen und `ON DELETE`-Konsequenzen abschätzen,
- `INNER JOIN` und `LEFT JOIN` richtig anwenden und den Unterschied benennen,
- `GROUP BY` zusammen mit den Aggregatfunktionen `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` lesen und schreiben,
- den Unterschied zwischen `WHERE` und `HAVING` sauber erklären,
- einfache **Subqueries**, **Views** und **Indizes** erkennen und verstehen, wofür sie gut sind,
- **Transaktionen** mit `BEGIN`/`COMMIT`/`ROLLBACK` einsetzen und die ACID-Eigenschaften benennen.

## ⏱️ Zeitbudget
Ca. 35 Minuten.

## 🧭 TL;DR
In V19 haben wir einzelne Tabellen erzeugt und abgefragt. Jetzt verbinden wir **mehrere** Tabellen über Fremdschlüssel, fassen Ergebnisse mit `GROUP BY` zusammen und lernen Werkzeuge kennen, mit denen echte Produktions-Datenbanken arbeiten: Views, Indizes und Transaktionen.

## 📦 Voraussetzungen
- V19 (Tabellen, `SELECT`/`INSERT`/`UPDATE`/`DELETE`, `WHERE`, Parameter-Binding).
- `00_python_recap.ipynb` dieser Vorlesung.


## 1. Wo wir stehen – und was jetzt kommt

In V19 hast du mit **einer** Tabelle gearbeitet – zum Beispiel `Produkte` mit `Produkt_ID`, `Name`, `Produktionszeit`. Alle Informationen zu einem Produkt lagen in einer einzigen Zeile, und die Welt war einfach.

In der Praxis hat man aber nie nur eine Tabelle. Jede Maschine hat viele Wartungen, jede Wartung gehört zu einem Techniker, jeder Techniker zu einer Abteilung. Wenn wir diese Information in eine einzige Tabelle pressen, entstehen **Redundanzen**: derselbe Techniker-Name steht hundertfach in der Wartungs-Tabelle, und wenn er heiratet, müssen wir hundert Zeilen ändern. Genau dort setzt dieses Notebook an.

## 2. Normalisierung – der Grundgedanke

**Normalisierung** ist die systematische Zerlegung einer großen, breiten Tabelle in mehrere kleine, schmalere Tabellen, sodass jede Information nur einmal gespeichert wird. Die drei üblichen Normalformen 1NF, 2NF und 3NF bauen aufeinander auf und wurden in den 1970er Jahren von Edgar F. Codd formuliert, dem Erfinder des relationalen Modells.

Wir schauen sie nur kurz an, damit du die Grundidee erkennst: **Jede Information gehört genau an einen Ort.** Wer das einhält, hat später weniger Ärger mit Inkonsistenzen, Tippfehlern und vergessenen Änderungen.

### 2.1 Erste Normalform (1NF)

Eine Tabelle ist in **1NF**, wenn jede Zelle genau **einen atomaren Wert** enthält. Listen oder komma-getrennte Felder wie `"Teil A, Teil B, Teil C"` in einer einzigen Spalte sind verboten; sie müssen in eigene Zeilen oder eigene Tabellen ausgelagert werden. Der Hintergedanke ist, dass man sonst keine saubere `WHERE`-Suche nach einzelnen Teilen formulieren kann.

### 2.2 Zweite Normalform (2NF)

Eine Tabelle ist in **2NF**, wenn sie zusätzlich zur 1NF die Eigenschaft erfüllt, dass jedes Nicht-Schlüssel-Attribut vom **gesamten** Primärschlüssel abhängt – nicht nur von einem Teil davon. Diese Forderung greift vor allem, wenn der Primärschlüssel aus mehreren Spalten zusammengesetzt ist.

Beispiel: In einer Tabelle `Bestellung (Bestell_ID, Produkt_ID, Produktname, Menge)` wäre `Produktname` bereits durch `Produkt_ID` allein festgelegt. Solche „teil-abhängigen" Spalten gehören in eine eigene Tabelle `Produkte`.

### 2.3 Dritte Normalform (3NF)

Eine Tabelle ist in **3NF**, wenn sie in 2NF ist und zusätzlich keine **transitiven Abhängigkeiten** enthält. Konkret heißt das: Ein Nicht-Schlüssel-Attribut darf nicht indirekt über ein anderes Nicht-Schlüssel-Attribut vom Schlüssel abhängen.

Stünde in `Maschinen (Maschinen_ID, Name, Abteilung, Abteilungsleiter)` der Abteilungsleiter nur deshalb drin, weil er zur Abteilung gehört, dann ist das eine transitive Abhängigkeit: `Maschinen_ID → Abteilung → Abteilungsleiter`. Die Lösung: eine eigene Tabelle `Abteilungen (Abteilung, Abteilungsleiter)`.

> [!NOTE]
> **Faustregel:** 3NF ist für unsere Zwecke „gut genug". Es gibt noch Boyce-Codd-Normalform und 4NF/5NF – für den Ingenieur-Alltag reicht 3NF in über 95 % der Fälle.

## 3. Fremdschlüssel (Foreign Key)

Sobald wir mehrere Tabellen haben, brauchen wir Verweise zwischen ihnen. Das ist die Aufgabe eines **Fremdschlüssels**. Ein Fremdschlüssel ist eine Spalte (oder Spaltenkombination) in einer Tabelle A, deren Werte auf den **Primärschlüssel** einer Tabelle B verweisen.

Durch diesen Mechanismus stellt die Datenbank **referentielle Integrität** sicher: Man kann nicht versehentlich eine Wartung für Maschine 99 eintragen, wenn es Maschine 99 gar nicht gibt. Der Fremdschlüssel wird mit `FOREIGN KEY (spalte) REFERENCES ziel_tabelle(ziel_spalte)` deklariert.

In [1]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("""
    CREATE TABLE Maschinen (
        Maschinen_ID INTEGER PRIMARY KEY,
        Name         TEXT
    )
""")
cur.execute("""
    CREATE TABLE Wartungen (
        Wartung_ID   INTEGER PRIMARY KEY,
        Maschinen_ID INTEGER,
        Datum        TEXT,
        FOREIGN KEY (Maschinen_ID) REFERENCES Maschinen(Maschinen_ID)
    )
""")
print("Tabellen angelegt.")
conn.close()


Tabellen angelegt.


> [!WARNING]
> SQLite prüft Fremdschlüssel standardmäßig **nicht**! Erst nach `PRAGMA foreign_keys = ON;` pro Verbindung werden die Einschränkungen wirklich durchgesetzt. Für unsere Notebooks ist das oft nebensächlich, aber in produktiven Anwendungen ist es Pflicht.

### 3.1 `ON DELETE` – was passiert, wenn die Quelle verschwindet?

Beim Anlegen des Fremdschlüssels kann man festlegen, wie die Datenbank reagiert, wenn eine referenzierte Zeile gelöscht wird. Die häufigsten Varianten sind `ON DELETE CASCADE` (alle abhängigen Zeilen werden mitgelöscht), `ON DELETE SET NULL` (die Fremdschlüssel-Spalte wird auf `NULL` gesetzt) und `ON DELETE RESTRICT` (das Löschen wird abgelehnt, solange noch etwas referenziert). Welches Verhalten sinnvoll ist, hängt von der Domäne ab: Für Wartungen zu einer verschrotteten Maschine ist `CASCADE` meist richtig, für Rechnungen zu einem Kunden meist nicht.

## 4. JOINs – mehrere Tabellen zusammenführen

Ein **JOIN** ist die SQL-Operation, die Zeilen zweier Tabellen anhand einer Bedingung kombiniert – fast immer entlang eines Fremdschlüssels. Das Ergebnis ist wieder eine Tabelle, in der Spalten beider Quelltabellen nebeneinander stehen.

Die beiden wichtigsten Varianten sind `INNER JOIN` und `LEFT JOIN`. Sie unterscheiden sich nur in einer einzigen, aber entscheidenden Frage: **Was passiert mit Zeilen, für die es keinen Partner in der anderen Tabelle gibt?**

### 4.1 `INNER JOIN`

Ein `INNER JOIN` liefert **nur** die Kombinationen, bei denen in beiden Tabellen ein passender Eintrag existiert. Alles ohne Gegenstück fällt weg. Anschaulich: Es ist die **Schnittmenge** der beiden Tabellen entlang der Join-Bedingung.

In [2]:
from IPython.display import Markdown, display
with open("diagramme/01_inner_join.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))


```mermaid
flowchart LR
    subgraph M["Tabelle Maschinen"]
        M1[ID=1 CNC-Fräse 01]
        M2[ID=2 CNC-Fräse 02]
        M3[ID=3 Drehmaschine 01]
    end
    subgraph W["Tabelle Wartungen"]
        W1[Wartung_1 Maschinen_ID=1]
        W2[Wartung_2 Maschinen_ID=1]
        W3[Wartung_3 Maschinen_ID=3]
        W4[Wartung_4 Maschinen_ID=99]:::orphan
    end
    M1 --- W1
    M1 --- W2
    M3 --- W3
    classDef orphan fill:#fee,stroke:#c00
    W4 -. verworfen .-> X((nicht im JOIN))

```

Syntaktisch ist ein INNER JOIN eine Erweiterung von `FROM`:

```sql
SELECT M.Name, W.Wartungstyp, W.Kosten
FROM Maschinen M
INNER JOIN Wartungen W ON M.Maschinen_ID = W.Maschinen_ID;
```

Hier werden für jede Kombination aus passender Maschine und Wartung eine neue Ergebniszeile gebildet. Die Aliase `M` und `W` machen den Code nur kürzer lesbar, sie sind nicht verpflichtend.

### 4.2 `LEFT JOIN`

Ein `LEFT JOIN` liefert **alle** Zeilen der linken Tabelle. Wo rechts kein passender Eintrag existiert, werden die rechten Spalten mit `NULL` aufgefüllt. Damit kann man zum Beispiel beantworten: *„Welche Maschinen haben noch nie eine Wartung erhalten?"* – das wäre mit `INNER JOIN` gar nicht möglich, weil genau diese Maschinen wegfallen würden.

In [3]:
with open("diagramme/02_left_join.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))


```mermaid
flowchart LR
    subgraph M["Tabelle Maschinen (links)"]
        M1[ID=1]
        M2[ID=2]
        M3[ID=5 Roboter]
        M6[ID=6 ohne Wartung]:::nowart
    end
    subgraph W["Tabelle Wartungen (rechts)"]
        W1[Maschinen_ID=1]
        W2[Maschinen_ID=2]
        W3[Maschinen_ID=5]
    end
    M1 --- W1
    M2 --- W2
    M3 --- W3
    M6 -. LEFT JOIN .-> N[NULL-Zeile]
    classDef nowart fill:#efe,stroke:#080

```

```sql
SELECT M.Name, W.Wartungstyp
FROM Maschinen M
LEFT JOIN Wartungen W ON M.Maschinen_ID = W.Maschinen_ID
WHERE W.Wartung_ID IS NULL;
```

Der Trick hier ist die `WHERE W.Wartung_ID IS NULL`-Klausel: Sie filtert gezielt jene Zeilen heraus, für die der LEFT JOIN kein Gegenstück gefunden hat. Dieses Muster werden wir in Aufgabe 5 verwenden.

### 4.3 `RIGHT JOIN` und `FULL OUTER JOIN` – nur konzeptuell

Ein `RIGHT JOIN` ist spiegelbildlich zum `LEFT JOIN`: Er behält alle Zeilen der rechten Tabelle. Da man jeden RIGHT JOIN durch Vertauschen der Tabellen als LEFT JOIN schreiben kann, wird er in der Praxis selten verwendet.

Ein `FULL OUTER JOIN` behält Zeilen aus **beiden** Seiten, auch wenn sie keinen Partner haben. SQLite unterstützt diesen erst seit Version 3.39 (Juni 2022). In MySQL fehlt er bis heute und muss durch `UNION` zweier JOINs emuliert werden.

## 5. `GROUP BY` und Aggregatfunktionen

Bisher haben wir immer einzelne Zeilen zurückbekommen. `GROUP BY` ändert das fundamental: Die Datenbank bildet **Gruppen** von Zeilen mit gleichem Wert in einer Gruppierungs-Spalte und liefert **pro Gruppe eine Ergebniszeile**. Zusammen mit den **Aggregatfunktionen** `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` kann man so Kennzahlen pro Gruppe berechnen.

In [4]:
with open("diagramme/03_group_by.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))


```mermaid
flowchart TD
    A[Rohdaten: 8 Wartungs-Zeilen] --> B[GROUP BY Maschinen_ID]
    B --> G1[Gruppe Maschinen_ID=1<br/>2 Zeilen]
    B --> G2[Gruppe Maschinen_ID=2<br/>1 Zeile]
    B --> G3[Gruppe Maschinen_ID=3<br/>3 Zeilen]
    B --> G4[Gruppe Maschinen_ID=4<br/>1 Zeile]
    B --> G5[Gruppe Maschinen_ID=5<br/>1 Zeile]
    G1 --> S1[SUM Kosten = 1350]
    G2 --> S2[SUM Kosten = 180]
    G3 --> S3[SUM Kosten = 2790]
    G4 --> S4[SUM Kosten = 160]
    G5 --> S5[SUM Kosten = 450]

```

### 5.1 Die fünf wichtigsten Aggregatfunktionen

| Funktion   | Bedeutung                                     |
|-----------|-----------------------------------------------|
| `COUNT(*)` | Anzahl Zeilen in der Gruppe                  |
| `SUM(x)`   | Summe der Werte in Spalte `x`                |
| `AVG(x)`   | arithmetischer Mittelwert                    |
| `MIN(x)`   | kleinster Wert                               |
| `MAX(x)`   | größter Wert                                 |

Diese Funktionen verlangen, dass jede **andere** Spalte in der `SELECT`-Liste entweder in `GROUP BY` auftaucht oder selbst aggregiert ist. Sonst ist unklar, welchen Wert die Datenbank liefern soll.

### 5.2 Ein komplettes GROUP-BY-Beispiel

```sql
SELECT Wartungstyp, COUNT(*) AS anzahl, AVG(Kosten) AS mittel
FROM Wartungen
GROUP BY Wartungstyp
ORDER BY anzahl DESC;
```

Diese Abfrage liefert für jeden Wartungstyp die Anzahl der Einträge und die durchschnittlichen Kosten. `AS anzahl` vergibt einen **Alias**, damit man die Spalte im `ORDER BY` – und später in Python – bequem ansprechen kann.

## 6. `WHERE` vs. `HAVING`

`WHERE` und `HAVING` filtern beide Zeilen heraus, arbeiten aber an **unterschiedlichen Stellen** der Ausführungskette:

- `WHERE` filtert **vor** der Gruppierung. Es sieht einzelne Original-Zeilen und kann keine Aggregate nutzen.
- `HAVING` filtert **nach** der Gruppierung. Es sieht bereits berechnete Gruppen mit ihren Aggregaten und darf auf `SUM`, `COUNT` etc. zugreifen.

In [5]:
with open("diagramme/04_having_vs_where.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))


```mermaid
flowchart TD
    A[Alle Zeilen] --> B{WHERE<br/>filtert Einzelzeilen}
    B -->|passt| C[Uebrige Zeilen]
    B -->|passt nicht| X[verworfen]
    C --> D[GROUP BY]
    D --> E[Gruppen mit Aggregaten]
    E --> F{HAVING<br/>filtert Gruppen}
    F -->|Gruppenbedingung ok| G[Ergebnis]
    F -->|Gruppe verworfen| Y[verworfen]

```

### 6.1 Konkret

```sql
-- Nur Wartungen aus dem Jahr 2024, dann gruppieren, dann nur
-- Maschinen mit Gesamtkosten > 1000 behalten:
SELECT Maschinen_ID, SUM(Kosten) AS gesamt
FROM Wartungen
WHERE Datum LIKE '2024-%'          -- Einzelzeilen-Filter
GROUP BY Maschinen_ID
HAVING gesamt > 1000;              -- Gruppen-Filter
```

Wer eine aggregierte Bedingung in `WHERE` schreibt, bekommt von SQLite einen Fehler: Die Aggregate existieren an dieser Stelle noch gar nicht.

> [!WARNING]
> Die Versuchung, einfach immer `HAVING` zu nehmen, ist groß – es scheint ja „mehr zu können". Tu das nicht: `WHERE` wird **vor** der Gruppierung ausgeführt und kann Indizes nutzen. Das ist bei großen Tabellen ein Riesenunterschied in der Laufzeit.

## 7. Subqueries – Abfragen in Abfragen

Eine **Subquery** ist ein `SELECT`, das innerhalb einer anderen SQL-Anweisung steht. Subqueries erlauben es, ein Zwischenergebnis zu berechnen und direkt weiterzuverwenden, ohne es in einer temporären Tabelle zu speichern.

### 7.1 `WHERE ... IN (SELECT ...)`

```sql
SELECT Name
FROM Maschinen
WHERE Maschinen_ID IN (
    SELECT Maschinen_ID FROM Wartungen WHERE Kosten > 1000
);
```

Die innere Abfrage liefert eine Liste von IDs, die äußere filtert darauf. Gleich nützlich lässt sich das Muster auch mit `NOT IN` einsetzen – etwa um Maschinen zu finden, zu denen es **keine** Wartung mit teurer Reparatur gibt.

### 7.2 Skalare Subquery

Eine **skalare Subquery** liefert genau einen einzelnen Wert und kann überall stehen, wo ein Wert erwartet wird – etwa im `SELECT` oder in einer `WHERE`-Bedingung:

```sql
SELECT Name
FROM Maschinen
WHERE Maschinen_ID = (
    SELECT Maschinen_ID FROM Wartungen
    ORDER BY Kosten DESC LIMIT 1
);
```

Das liefert die Maschine mit der **teuersten einzelnen** Wartung. Wichtig: Liefert die innere Abfrage mehr als eine Zeile, bricht SQLite mit einem Fehler ab.

## 8. Views – gespeicherte Abfragen

Ein **View** ist eine unter einem Namen abgelegte SELECT-Abfrage. Aus Sicht des Nutzers verhält er sich wie eine Tabelle: Man kann `SELECT * FROM view_name` schreiben, obwohl hinter den Kulissen jedes Mal die gespeicherte Abfrage ausgeführt wird.

Views sind nützlich, um oft gebrauchte Abfragen nicht ständig neu zu tippen und um komplexe SQL vor den Endanwendern zu verbergen. Sie werden mit `CREATE VIEW name AS SELECT ...` angelegt.

```sql
CREATE VIEW Wartungsuebersicht AS
SELECT M.Name, W.Wartungstyp, W.Kosten
FROM Maschinen M
INNER JOIN Wartungen W ON M.Maschinen_ID = W.Maschinen_ID;

-- Jetzt kann man einfach abfragen:
SELECT * FROM Wartungsuebersicht WHERE Kosten > 500;
```

## 9. Indizes – Beschleuniger für große Tabellen

Ein **Index** ist eine zusätzliche Datenstruktur, die die Datenbank anlegt, damit bestimmte Abfragen schneller werden. Ohne Index muss die DB bei einer `WHERE`-Suche jede Zeile anschauen (**Full Table Scan**); mit einem passenden Index – meist ein B-Baum – findet sie die gesuchten Zeilen in logarithmischer Zeit.

Der Primärschlüssel erhält automatisch einen Index. Für zusätzliche Spalten, nach denen häufig gefiltert oder gejoint wird, kann man mit `CREATE INDEX` nachrüsten:

```sql
CREATE INDEX idx_wartungen_maschine ON Wartungen(Maschinen_ID);
```

Indizes sind aber nicht kostenlos: Sie brauchen Speicher und müssen bei jedem `INSERT`/`UPDATE`/`DELETE` mitgepflegt werden. Für kleine Tabellen oder reine Lesetabellen ist das meist egal; bei Millionen Zeilen wird es zur Designfrage.

## 10. Transaktionen und ACID

Eine **Transaktion** bündelt mehrere SQL-Anweisungen zu einer logischen Einheit, die entweder **komplett** ausgeführt oder **gar nicht** wirksam wird. Das ist zum Beispiel beim Umbuchen von Geld zwischen zwei Konten lebenswichtig: Entweder wird bei A abgebucht **und** bei B gutgeschrieben – oder nichts passiert.

### 10.1 Die ACID-Eigenschaften

- **Atomicity (Atomarität):** Alle Anweisungen der Transaktion wirken zusammen oder gar nicht.
- **Consistency (Konsistenz):** Nach der Transaktion ist die DB wieder in einem gültigen Zustand (alle Constraints erfüllt).
- **Isolation (Isolation):** Parallele Transaktionen beeinflussen sich nicht sichtbar – jede sieht eine konsistente Version der Daten.
- **Durability (Dauerhaftigkeit):** Nach `COMMIT` sind die Änderungen auch einen Stromausfall später noch da.

### 10.2 `BEGIN` / `COMMIT` / `ROLLBACK`

In SQL startet man eine Transaktion mit `BEGIN`, beendet sie erfolgreich mit `COMMIT` oder verwirft sie mit `ROLLBACK`. In Python mit `sqlite3` ist das meist eleganter über den `with`-Kontext der Verbindung: Bei erfolgreichem Block-Ende wird automatisch committet, bei einer Exception automatisch zurückgerollt.

In [6]:
import sqlite3
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE konten (id INTEGER PRIMARY KEY, saldo REAL)")
cur.executemany("INSERT INTO konten VALUES (?, ?)", [(1, 1000.0), (2, 500.0)])
conn.commit()

try:
    with conn:  # automatisches BEGIN / COMMIT / ROLLBACK
        cur.execute("UPDATE konten SET saldo = saldo - 200 WHERE id = 1")
        cur.execute("UPDATE konten SET saldo = saldo + 200 WHERE id = 2")
except Exception:
    print("Fehler – automatisch zurückgerollt")

cur.execute("SELECT id, saldo FROM konten ORDER BY id")
print(cur.fetchall())
conn.close()


[(1, 800.0), (2, 700.0)]


## 11. Maschinenbau-Beispiel – alles auf einmal

Angenommen, die Fertigungsleitung fragt: *„Welche Maschine hat uns im letzten Jahr am meisten Wartungskosten verursacht?"* Die passende SQL kombiniert genau die Bausteine, die wir gerade kennengelernt haben:

```sql
SELECT M.Name, SUM(W.Kosten) AS gesamt
FROM Maschinen M
INNER JOIN Wartungen W ON M.Maschinen_ID = W.Maschinen_ID
GROUP BY M.Maschinen_ID
ORDER BY gesamt DESC
LIMIT 1;
```

Der INNER JOIN holt Name und Wartungskosten zusammen, `GROUP BY` macht aus vielen Wartungszeilen pro Maschine eine einzige Zeile, `SUM` addiert die Kosten, `ORDER BY ... DESC LIMIT 1` liefert den teuersten Fall. **Dieses Muster** (JOIN → GROUP BY → ORDER BY → LIMIT) wirst du in Aufgabe 4 wiedersehen.

## 12. Selbstcheck

<details><summary>❓ Wie unterscheiden sich `INNER JOIN` und `LEFT JOIN`?</summary>

`INNER JOIN` liefert nur Zeilen, die in beiden Tabellen Partner haben. `LEFT JOIN` behält **alle** Zeilen der linken Tabelle; fehlt rechts ein Partner, stehen dort `NULL`-Werte.

</details>

<details><summary>❓ Warum darf man `COUNT(*) > 5` nicht in `WHERE` schreiben?</summary>

`WHERE` filtert einzelne Zeilen **vor** der Gruppierung, da existiert `COUNT(*)` noch gar nicht. Solche Bedingungen gehören in `HAVING`, das nach `GROUP BY` greift.

</details>

<details><summary>❓ Was bringt ein Index auf einer Spalte?</summary>

Er beschleunigt `WHERE`-Suchen und JOINs auf dieser Spalte von linearer zu logarithmischer Komplexität, kostet aber Speicher und macht Schreiboperationen etwas langsamer.

</details>

<details><summary>❓ Wozu dient ein Fremdschlüssel?</summary>

Er stellt sicher, dass jeder Verweis in einer Tabelle auf einen existierenden Eintrag in einer anderen Tabelle zeigt – das nennt man referentielle Integrität.

</details>

<details><summary>❓ Was bedeutet die „A" in ACID?</summary>

Atomarität: Eine Transaktion wird entweder komplett wirksam oder gar nicht – niemals halb.

</details>

<details><summary>❓ Wann nehme ich einen View?</summary>

Wenn dieselbe komplizierte Abfrage öfter gebraucht wird und man sie unter einem sprechenden Namen einmal definieren möchte, statt sie überall neu zu tippen.

</details>

<details><summary>❓ Warum braucht man die 3NF?</summary>

Sie vermeidet transitive Abhängigkeiten. Dadurch verhindert sie, dass dieselbe Information (z. B. der Abteilungsleiter) in mehreren Zeilen auftaucht und bei Änderungen inkonsistent wird.

</details>

<details><summary>❓ Was macht `HAVING gesamt > 1000` nach `GROUP BY`?</summary>

Es behält nur jene Gruppen, deren aggregierter Wert `gesamt` größer als 1000 ist; alle anderen Gruppen verschwinden aus dem Ergebnis.

</details>


> [!TIP]
> **Weiterführend:** Für sehr große Analysen setzt man statt Subqueries häufig **CTEs** (`WITH name AS (...)`) ein, weil sie lesbarer sind. SQLite unterstützt CTEs seit Version 3.8. In Kurs und Praxis reichen Subqueries aber völlig aus.

## ✅ Zusammenfassung
- **Normalisierung** bis 3NF vermeidet Redundanz und Inkonsistenz.
- **Fremdschlüssel** verknüpfen Tabellen; `ON DELETE CASCADE/SET NULL/RESTRICT` steuert die Reaktion auf Löschungen.
- `INNER JOIN` = Schnittmenge, `LEFT JOIN` = alles links + NULLs rechts.
- `GROUP BY` plus Aggregate `COUNT/SUM/AVG/MIN/MAX` liefern Kennzahlen pro Gruppe.
- `WHERE` filtert Zeilen, `HAVING` filtert Gruppen – der Unterschied ist die Ausführungsreihenfolge.
- Subqueries, Views und Indizes sind Werkzeuge der echten Praxis.
- Transaktionen (`BEGIN/COMMIT/ROLLBACK`) garantieren ACID.

## ➡️ Nächster Schritt
Weiter mit [02_praxis.ipynb](02_praxis.ipynb).
